# Chương 3 bài 1 SweetViz trên Marketing Campaign

**Nguồn:** Lab EDA, mục 1.3.3, trang 18

**Đề bài:** Tìm hiểu tính năng, cách sử dụng SweetViz và áp dụng trên Marketing Campaign.

Code triển khai bài tập theo dữ liệu đã cung cấp; không phải đoạn chép nguyên văn bài mẫu.

In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data').is_dir() and (p / 'README.md').is_file())
DATA = ROOT / 'data'
OUT = ROOT / 'ket_qua' / 'C3_BT01_SweetViz_Marketing'
OUT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (8, 4)})

def save_fig(name):
    plt.tight_layout()
    plt.savefig(OUT / name, dpi=140, bbox_inches='tight')
    plt.show()

def save_summary(values):
    (OUT / 'tom_tat.json').write_text(
        json.dumps(values, ensure_ascii=False, indent=2), encoding='utf-8'
    )


## 1 Chuẩn bị dữ liệu

In [2]:
marketing = pd.read_csv(DATA / 'marketing_campaign.csv', sep=None, engine='python')
print('Shape:', marketing.shape)
print('Missing values:', marketing.isna().sum()[marketing.isna().any()].to_dict())
print('Response:', marketing['Response'].value_counts().to_dict())
# Historical ages are referenced to the latest customer-registration year.
joined = pd.to_datetime(marketing['Dt_Customer'], dayfirst=True, errors='coerce')
reference_year = int(joined.dt.year.max())
marketing['Age_at_reference'] = reference_year - marketing['Year_Birth']
marketing['Customer_Join_Year'] = joined.dt.year
marketing = marketing.drop(columns=['ID', 'Dt_Customer'])
constant_cols = [c for c in marketing if marketing[c].nunique(dropna=False) <= 1]
marketing = marketing.drop(columns=constant_cols)
print('Reference year:', reference_year)
print('Constant columns excluded:', constant_cols)
print('Age > 100 (flag for review):', int((marketing['Age_at_reference'] > 100).sum()))
# Preserve missing Income and outliers for EDA diagnostics, not automatic deletion.

Shape: (2240, 29)
Missing values: {'Income': 24}
Response: {0: 1906, 1: 334}
Reference year: 2014
Constant columns excluded: ['Z_CostContact', 'Z_Revenue']
Age > 100 (flag for review): 3


## 2 Sinh báo cáo SweetViz

In [3]:
import sweetviz as sv
report = sv.analyze(marketing, target_feat='Response', pairwise_analysis='on')
report_path = OUT / 'SweetViz_Marketing.html'
report.show_html(str(report_path), open_browser=False)
print('Report:', report_path.relative_to(ROOT))
print('HTML size:', report_path.stat().st_size)
save_summary({'rows':len(marketing), 'analyzed_columns':marketing.shape[1],
              'reference_year':reference_year,
              'missing_income':int(marketing.Income.isna().sum()),
              'response_rate':float(marketing.Response.mean()),
              'age_over_100':int((marketing.Age_at_reference>100).sum()),
              'html_file':'SweetViz_Marketing.html', 'html_bytes':report_path.stat().st_size})

Report /workspace/scratch/f6741a6f96bb/output/BoSung_BaiTap_2_Lab/ket_qua/C3_BT01_SweetViz_Marketing/SweetViz_Marketing.html was generated.
Report: ket_qua/C3_BT01_SweetViz_Marketing/SweetViz_Marketing.html
HTML size: 2755507


/workspace/scratch/f6741a6f96bb/work/lab_env/lib/python3.12/site-packages/sweetviz/graph.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)


## Nhận xét và giới hạn
SweetViz tạo báo cáo HTML gồm loại biến, phân bố, thiếu dữ liệu và liên hệ với Response. Mở HTML trong trình duyệt để xem. ID và cột hằng được loại khỏi phân tích, ngày được biến đổi thành năm tham gia; Income thiếu và tuổi bất thường vẫn được giữ để phát hiện vấn đề. Báo cáo tự động không thay thế diễn giải nghiệp vụ hoặc chứng minh nhân quả. Tham khảo API chính thức: https://github.com/fbdesignpro/sweetviz